# LSTM — Future 30s Apnea Event Detection

Mirrors the LightGBM pipeline in `1_1-tree-based-models.ipynb`.
Key difference: instead of flattening lag features into a tabular row,
we feed a **proper sequence of T windows** to a unidirectional LSTM.

Input shape: `(batch, T, n_features)` where
- `T` = sequence length (number of 5s windows of lookback context)
- `n_features` = all features (excluding patient metadata), z-scored per-patient on train stats

In [ ]:
from __future__ import annotations

import json
import math
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
import optuna

from src.data_prepatation import create_batched_splits
from config import dir_config

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
compiled_dir = dir_config.data.compiled
processed_dir = dir_config.data.processed

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

random_seed = 2542
torch.manual_seed(random_seed)
np.random.seed(random_seed)

## 1. Load data — same splits as LightGBM baseline

In [ ]:
parquet_files = list(Path(processed_dir).glob('*_with_metadata.parquet'))
parquet_files = [f for f in parquet_files if f.name != 'agg_data_with_metadata.parquet']

target_future_step = 5   # 5 × 5s = 25s lookahead (same as LightGBM notebook)

# Step 1 — Load flat splits (no lags yet; we build sequences ourselves)
train_X, train_y, val_X, val_y, test_X, test_y, left_out = create_batched_splits(
    parquet_files,
    batch_size=360,          # 30-min chunks
    gap_size=6,              # 30s gap before label window
    top_features=None,
    top_features_lag=0,
    target_type='apnea',
    target_future_steps=target_future_step,
    val_ratio=0.2,
    test_ratio=0.2,
    n_leave_out=5,
    random_seed=random_seed,
)

print(f'Train: {train_X.shape}, Val: {val_X.shape}, Test: {test_X.shape}')
print(f'Train pos rate: {train_y.mean():.3f}')

## 2. Build per-patient sequence dataset

Each sample is a sliding window of **T consecutive 5s epochs** ending at row i,
with the label being whether an apnea event occurs in the 30s *after* the sequence.

**Important design choices:**
- Sequences never cross patient or chunk boundaries (avoids data leakage)
- Features are z-scored **per patient**, using only train-set mean/std
- `SEQ_LEN` is a hyperparameter; start at 12 (60s), also try 18 and 24

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
SEQ_LEN = 12        # number of 5s windows per sequence = 60s lookback
                    # also try 18 (90s) and 24 (120s)

# Use all features, excluding patient metadata columns
subject_metadata = pd.read_csv(Path(processed_dir) / 'participant_info.csv')
metadata_cols = set(subject_metadata.columns)
metadata_cols.add('chunk_id')  # also exclude our chunk identifier column
metadata_cols.add('subject_id')  # also exclude subject identifier column
FEATURES = [col for col in train_X.columns if col not in metadata_cols]

N_FEATURES = len(FEATURES)
print(f'Using {N_FEATURES} features')

# ── Per-patient z-score normalisation ───────────────────────────────────────
# Must be fit on train set only; apply same stats to val/test
def compute_patient_stats(
    X: pd.DataFrame, features: List[str]
) -> Dict[str, Tuple[pd.Series, pd.Series]]:
    """Returns {patient_id: (mean, std)} for each patient in X."""
    stats = {}
    if 'subject_id' in X.columns:
        grp_col = 'subject_id'
    else:
        mu  = X[features].mean()
        std = X[features].std().replace(0, 1).fillna(1)
        return {'__global__': (mu, std)}

    for pid, grp in X.groupby(grp_col):
        mu  = grp[features].mean()
        std = grp[features].std().replace(0, 1).fillna(1)  # fillna(1) handles all-NaN columns
        stats[pid] = (mu, std)
    return stats


def apply_patient_stats(
    X: pd.DataFrame,
    features: List[str],
    train_stats: Dict,
) -> pd.DataFrame:
    """Z-score X using train_stats. Unknown patients get global train mean/std."""
    X = X.copy()
    if '__global__' in train_stats:
        mu, std = train_stats['__global__']
        X[features] = (X[features] - mu) / std
        X[features] = X[features].fillna(0)  # fill any remaining NaN with 0
        return X

    grp_col = 'subject_id'
    # Global fallback stats (for patients not in train)
    all_mu  = pd.concat([s[0] for s in train_stats.values()], axis=1).mean(axis=1)
    all_std = pd.concat([s[1] for s in train_stats.values()], axis=1).mean(axis=1)

    for pid, idx in X.groupby(grp_col).groups.items():
        mu, std = train_stats.get(pid, (all_mu, all_std))
        X.loc[idx, features] = (X.loc[idx, features] - mu) / std

    X[features] = X[features].fillna(0)  # fill any remaining NaN with 0
    return X


train_stats = compute_patient_stats(train_X, FEATURES)

train_X_norm = apply_patient_stats(train_X, FEATURES, train_stats)
val_X_norm   = apply_patient_stats(val_X,   FEATURES, train_stats)
test_X_norm  = apply_patient_stats(test_X,  FEATURES, train_stats)

print('Normalisation done.')
print(f'Train NaN count: {train_X_norm[FEATURES].isna().sum().sum()}')
print(f'Train mean (should be ~0): {train_X_norm[FEATURES].mean().mean():.4f}')
print(f'Train std  (should be ~1): {train_X_norm[FEATURES].std().mean():.4f}')


In [ ]:
class ApneaSequenceDataset(Dataset):
    """Sliding-window sequence dataset.

    Each item is:
        x : FloatTensor of shape (SEQ_LEN, N_FEATURES)
        y : FloatTensor scalar — apnea label for the window *after* this sequence

    Sequences are constructed within contiguous patient-chunk blocks so that
    a sequence never crosses a patient or a 30-min chunk boundary.

    Parameters
    ----------
    X : pd.DataFrame  — normalised feature matrix
    y : pd.Series     — binary labels (aligned with X)
    seq_len : int     — number of 5s windows per sequence
    features : list   — column names to include
    chunk_col : str   — column that identifies contiguous 30-min chunks
                        (adjust to match your actual column name)
    """

    def __init__(
        self,
        X: pd.DataFrame,
        y: pd.Series,
        seq_len: int,
        features: List[str],
        chunk_col: str = 'chunk_id',
    ):
        self.seq_len  = seq_len
        self.features = features
        self.samples: List[Tuple[np.ndarray, float]] = []

        # If there's no chunk_col, treat the whole DataFrame as one block
        if chunk_col not in X.columns:
            print(f"Warning: '{chunk_col}' not found — treating data as one block. "
                  f"Set chunk_col to your actual chunk identifier.")
            self._extract_sequences(X, y)
        else:
            for _, chunk_idx in X.groupby(chunk_col).groups.items():
                chunk_X = X.loc[chunk_idx]
                chunk_y = y.loc[chunk_idx]
                self._extract_sequences(chunk_X, chunk_y)

        print(f'  {len(self.samples):,} sequences | pos rate: '
              f'{np.mean([s[1] for s in self.samples]):.3f}')

    def _extract_sequences(self, X: pd.DataFrame, y: pd.Series) -> None:
        vals = X[self.features].values.astype(np.float32)
        labs = y.values.astype(np.float32)
        n = len(vals)
        # Sequence [i : i+seq_len] predicts label at position i+seq_len
        for i in range(n - self.seq_len):
            self.samples.append((vals[i : i + self.seq_len], labs[i + self.seq_len]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, label = self.samples[idx]
        return torch.tensor(x), torch.tensor(label)


# ── Build datasets ───────────────────────────────────────────────────────────
# Set chunk_col to whatever column in your DataFrame identifies the 30-min chunks
# Common names: 'chunk_id', 'batch_id', 'window_id' — check train_X.columns
CHUNK_COL = 'chunk_id'   # <-- adjust if your column name differs

print('Building train dataset...')
train_ds = ApneaSequenceDataset(train_X_norm, train_y, SEQ_LEN, FEATURES, CHUNK_COL)
print('Building val dataset...')
val_ds   = ApneaSequenceDataset(val_X_norm,   val_y,   SEQ_LEN, FEATURES, CHUNK_COL)
print('Building test dataset...')
test_ds  = ApneaSequenceDataset(test_X_norm,  test_y,  SEQ_LEN, FEATURES, CHUNK_COL)

## 4. LSTM model

In [ ]:
class ApneaLSTM(nn.Module):
    """Unidirectional LSTM for future apnea event prediction.

    Architecture
    ------------
    Input (batch, T, n_features)
        → LSTM layers  (unidirectional — no future leakage)
        → final hidden state h_T
        → dropout
        → linear head → sigmoid

    Why final hidden state and not mean pooling?
    The model should weight recent context most heavily when predicting
    what happens *next* — h_T naturally encodes this recency bias.
    """

    def __init__(
        self,
        n_features: int,
        hidden_size: int = 64,
        num_layers: int = 2,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,  # nn.LSTM dropout only between layers
            bidirectional=False,   # MUST be False — predicting future events
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, T, n_features)
        _, (h_n, _) = self.lstm(x)
        # h_n: (num_layers, batch, hidden_size) — take the last layer
        last_hidden = h_n[-1]           # (batch, hidden_size)
        logit = self.head(last_hidden)  # (batch, 1)
        return logit.squeeze(1)         # (batch,)


# Quick sanity check
model = ApneaLSTM(n_features=N_FEATURES).to(DEVICE)
dummy = torch.zeros(4, SEQ_LEN, N_FEATURES).to(DEVICE)
out = model(dummy)
print(f'Output shape: {out.shape}  (expected: torch.Size([4]))')
print(f'Parameter count: {sum(p.numel() for p in model.parameters()):,}')

## 5. Training utilities

In [ ]:
class FocalLoss(nn.Module):
    """Focal loss — better than weighted BCE for skewed AUCPR optimisation.

    alpha : weight on the positive class (set to neg/pos ratio, or tune it)
    gamma : focusing parameter — higher = more focus on hard examples (try 1–3)
    """

    def __init__(self, alpha: float = 1.0, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        p_t = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        return (focal_weight * bce).mean()


def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimiser: torch.optim.Optimizer,
    loss_fn: nn.Module,
    device: torch.device,
) -> float:
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimiser.zero_grad()
        logits = model(x)
        loss   = loss_fn(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # stabilises LSTM training
        optimiser.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    threshold: float = 0.5,
) -> Dict:
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        x = x.to(device)
        probs = torch.sigmoid(model(x)).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())

    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    preds  = (probs >= threshold).astype(int)

    return {
        'auc_pr':  average_precision_score(labels, probs),
        'auc_roc': roc_auc_score(labels, probs),
        'f1':      f1_score(labels, preds, zero_division=0),
        'classification_report': classification_report(labels, preds, zero_division=0),
        'confusion_matrix': confusion_matrix(labels, preds),
    }

## 6. Optuna hyperparameter search

Mirrors the Optuna pattern from the LightGBM notebook.
Tunes: hidden size, num layers, dropout, learning rate, batch size, focal loss gamma.
Sequence length (`SEQ_LEN`) is NOT tuned here — run cells 3–6 with SEQ_LEN=12/18/24
and compare val AUCPR manually, as it requires rebuilding the dataset.

In [ ]:
neg_count = int((train_y == 0).sum())
pos_count = int((train_y == 1).sum())
pos_weight_ratio = neg_count / pos_count
print(f'Class imbalance ratio (neg/pos): {pos_weight_ratio:.2f}')


def make_loaders(batch_size: int) -> Tuple[DataLoader, DataLoader]:
    """Recreate loaders with a given batch size (needed inside Optuna trials)."""
    tr = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=True)
    va = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    return tr, va


def run_trial(
    hidden_size: int,
    num_layers: int,
    dropout: float,
    lr: float,
    batch_size: int,
    focal_alpha: float,
    focal_gamma: float,
    max_epochs: int = 30,
    patience: int = 5,
) -> Tuple[float, nn.Module]:
    """Train a model with given hyperparameters; return (best_val_aucpr, model)."""
    torch.manual_seed(random_seed)

    model = ApneaLSTM(
        n_features=N_FEATURES,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
    ).to(DEVICE)

    optimiser  = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimiser, mode='max', patience=3, factor=0.5, verbose=False
    )
    loss_fn    = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
    train_loader, val_loader = make_loaders(batch_size)

    best_aucpr = 0.0
    best_state = None
    no_improve = 0

    for epoch in range(max_epochs):
        train_loss = train_epoch(model, train_loader, optimiser, loss_fn, DEVICE)
        val_metrics = evaluate(model, val_loader, DEVICE)
        scheduler.step(val_metrics['auc_pr'])

        if val_metrics['auc_pr'] > best_aucpr:
            best_aucpr = val_metrics['auc_pr']
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_aucpr, model


def objective(trial: optuna.Trial) -> float:
    hidden_size  = trial.suggest_categorical('hidden_size', [32, 64, 128])
    num_layers   = trial.suggest_int('num_layers', 1, 3)
    dropout      = trial.suggest_float('dropout', 0.1, 0.5)
    lr           = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size   = trial.suggest_categorical('batch_size', [64, 128, 256])
    focal_alpha  = trial.suggest_float('focal_alpha', 0.3, 0.9)   # weight on positive class
    focal_gamma  = trial.suggest_float('focal_gamma', 1.0, 3.0)

    aucpr, _ = run_trial(
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
        lr=lr,
        batch_size=batch_size,
        focal_alpha=focal_alpha,
        focal_gamma=focal_gamma,
        max_epochs=30,
        patience=5,
    )
    return aucpr


study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=random_seed),
)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f'\nBest val AUC-PR: {study.best_value:.4f}')
print(f'Best params:     {study.best_params}')

## 7. Retrain best model and evaluate

In [ ]:
bp = study.best_params

print('Retraining best model with longer budget (max_epochs=100, patience=10)...')
best_aucpr, best_model = run_trial(
    hidden_size  = bp['hidden_size'],
    num_layers   = bp['num_layers'],
    dropout      = bp['dropout'],
    lr           = bp['lr'],
    batch_size   = bp['batch_size'],
    focal_alpha  = bp['focal_alpha'],
    focal_gamma  = bp['focal_gamma'],
    max_epochs   = 100,
    patience     = 10,
)

_, val_loader_eval  = make_loaders(bp['batch_size'])
test_loader = DataLoader(test_ds, batch_size=bp['batch_size'], shuffle=False, num_workers=0)

print('\n--- Final model evaluation ---')
for split_name, loader in [
    ('Train', DataLoader(train_ds, batch_size=bp['batch_size'], shuffle=False, num_workers=0)),
    ('Val',   val_loader_eval),
    ('Test',  test_loader),
]:
    m = evaluate(best_model, loader, DEVICE)
    print(f"{split_name:5s}  AUC-ROC={m['auc_roc']:.4f}  AUC-PR={m['auc_pr']:.4f}  F1={m['f1']:.4f}")

print('\nTest classification report:')
print(evaluate(best_model, test_loader, DEVICE)['classification_report'])

## 8. Save model and hyperparameters

In [ ]:
save_dir = Path(processed_dir) / 'models'
save_dir.mkdir(exist_ok=True)

torch.save(best_model.state_dict(), save_dir / 'lstm_best.pt')

run_config = {
    'seq_len':     SEQ_LEN,
    'features':    FEATURES,
    'best_params': bp,
    'best_val_aucpr': study.best_value,
}
with open(save_dir / 'lstm_config.json', 'w') as f:
    json.dump(run_config, f, indent=2)

print(f'Model saved to {save_dir / "lstm_best.pt"}')
print(f'Config saved to {save_dir / "lstm_config.json"}')

## 9. SEQ_LEN sweep (run after above is stable)

Re-run cells 3–8 with SEQ_LEN = 12, 18, 24 and compare val AUCPR.
This cell summarises the results if you store them manually.

In [ ]:
# Fill in after running the sweep
seq_len_results = {
    # seq_len: val_aucpr
    # 12: 0.xxx,
    # 18: 0.xxx,
    # 24: 0.xxx,
}

if seq_len_results:
    best_seq = max(seq_len_results, key=seq_len_results.get)
    print(f'Best SEQ_LEN: {best_seq} → val AUCPR = {seq_len_results[best_seq]:.4f}')
    for k, v in sorted(seq_len_results.items()):
        print(f'  SEQ_LEN={k:3d}  val AUC-PR={v:.4f}')
else:
    print('Run SEQ_LEN sweep and fill in seq_len_results above.')